In [1]:
#spark.stop()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("data_skew_homework").config("spark.sql.adaptive.enabled", "false").config("spark.sql.autoBroadcastJoinThreshold", "-1").config("spark.driver.memory", "15g").master("local[2]").getOrCreate()
#spark.conf.set("spark.sql.adaptive.enabled", "false")
#spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
# .config("spark.sql.shuffle.partitions", 100)
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/11 10:40:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
#spark.sparkContext.getConf().getAll()

### Read inputs

In [4]:
events_df = spark.read.json("events.json")
users_df = spark.read.csv("users.csv", header=True)

### Default join

In [5]:
from pyspark.sql.window import Window

events_users = events_df.join(users_df, on="user_id")
#events_users.write.format("noop").mode("append").save()

In [6]:
events_users = events_users.withColumn("event_date_num", F.datediff(F.current_date(), F.to_date("event_timestamp")))
events_last_90_days = Window.partitionBy("user_id").orderBy("event_date_num").rangeBetween(0, 90)
events_users = events_users.withColumn("rolling_90d_events", F.count("*").over(events_last_90_days))
events_users.write.csv("events_with_rolling.csv", header=True, mode="overwrite")

### Salting

In [7]:
from pyspark.sql.window import Window

SALT_FACTOR = 10  # standard salt factor

# 1. Prepare data with salt for Join
events_salted = events_df.withColumn("salt", (F.rand() * SALT_FACTOR).cast("int"))

users_salted = users_df.withColumn("salt", F.explode(F.array([F.lit(i) for i in range(SALT_FACTOR)])))

# Join by 2 columns instead of overhead with concat salted_key
joined_salted = events_salted.join(users_salted, on=["user_id", "salt"], how="inner")
#joined_salted.write.format("noop").mode("append").save()

In [8]:
# Add a numeric date representation for Window range evaluation
joined_salted = joined_salted.withColumn("event_date_num", F.datediff(F.current_date(), F.to_date("event_timestamp")))

# 2. Calculate the window within each salt
# partitionBy(user_id, salt) here is intentionally WRONG for a true rolling window —
# each salt bucket only sees a fraction of a user's own events, so rangeBetween(0, 90)
# is computed over an incomplete, randomly-sliced subset of that user's dates.
# Summing the partial per-salt counts below does NOT reconstruct the true rolling count
# (double-counts on date overlaps between salt buckets, undercounts on gaps).
# Salting is safe for equi-joins/plain aggregates, but not for order/range window functions.
# The correct fix for this is the "Separate treatment" section further down.
window_local = Window.partitionBy("user_id", "salt").orderBy("event_date_num").rangeBetween(0, 90)
df_local_aggr = joined_salted.withColumn("local_count", F.count("*").over(window_local))

# 3. Merging the salted results
# Group by original user_id and date to sum up metrics across all salts
df_final = df_local_aggr.groupBy("user_id", "event_date_num") \
                        .agg(F.sum("local_count").alias("rolling_90d_events"))

df_final.write.csv("events_with_rolling_salting.csv", header=True, mode="overwrite")

### Separate treatment

In [9]:
# 1. Find hot user_ids (top-N by event count)
N = 4

# Variant 1: hardcode an approximate threshold, no global sort
# THRESHOLD = 1000
# hot_users_df = events_df.groupBy("user_id").count().filter(F.col("count") > THRESHOLD)

# Variant 2: window function (row_number) over the aggregated counts.
# No partitionBy -> the whole aggregated counts_df is moved into a single partition
# to guarantee one consistent global ordering before numbering the rows.
# w = Window.orderBy(F.col("count").desc())
# hot_users_df = (
#     events_df.groupBy("user_id").count()
#     .withColumn("rn", F.row_number().over(w))
#     .filter(F.col("rn") <= N)
#     .drop("rn")
#     .cache()
# )

# Variant 3 (my old approach): orderBy + limit + collect() to build a Python list for isin().
# Leaves Spark entirely - pulls the top-N rows to the driver as Python objects, then
# embeds them as a literal IN-list in the next filter instead of staying a DataFrame.
# top_users_rows = events_df.groupBy("user_id").count().orderBy(F.col("count").desc()).limit(N).collect()
# hot_user_ids = [row["user_id"] for row in top_users_rows]
# events_hot = events_df.filter(F.col("user_id").isin(hot_user_ids))

# Chosen: exact top-N via orderBy + limit, kept as a DataFrame (no .collect(), stays distributed)
# Spark optimizes Sort+Limit into TakeOrderedAndProject: each partition keeps only its local top-N
# (bounded heap), and just N * numPartitions candidate rows are shuffled to the final merge —
# cheaper than the window approach above, which has to move the whole dataset into one partition.
hot_users_df = (
    events_df.groupBy("user_id")
    .count()
    .orderBy(F.col("count").desc())
    .limit(N)
    .cache()
)
#hot_users_df.show()

In [10]:
# 2. Split events into hot and cold parts
events_hot = events_df.join(hot_users_df.select("user_id"), on="user_id", how="inner")
events_cold = events_df.join(hot_users_df.select("user_id"), on="user_id", how="left_anti")

#print("Hot events:", events_hot.count())
#print("Cold events:", events_cold.count())

In [11]:
# 3. Cold part: standard path, no tricks needed (many distinct keys, naturally balanced)
joined_cold = events_cold.join(users_df, on="user_id")
joined_cold = joined_cold.withColumn("event_date_num", F.datediff(F.current_date(), F.to_date("event_timestamp")))
window_cold = Window.partitionBy("user_id").orderBy("event_date_num").rangeBetween(0, 90)
joined_cold = joined_cold.withColumn("rolling_90d_events", F.count("*").over(window_cold))

In [12]:
# 4. Hot part: pre-aggregate to user+day level (few keys, huge row counts)
# the split already narrows this down to a handful of unique user_id values — no salting needed here (salting typically isn't used after separation)
events_hot_with_date = events_hot.withColumn("event_date_num", F.datediff(F.current_date(), F.to_date("event_timestamp")))
daily_counts_hot = (
    events_hot_with_date.groupBy("user_id", "event_date_num")
    .count()
    .withColumnRenamed("count", "daily_count")
)

In [13]:
# 5. Rolling window on the now-small daily aggregates (no salt needed, data is tiny per user)
window_hot_daily = Window.partitionBy("user_id").orderBy("event_date_num").rangeBetween(0, 90)
daily_counts_hot = daily_counts_hot.withColumn("rolling_90d_events", F.sum("daily_count").over(window_hot_daily))

In [14]:
# 6. Join rolling result back to original hot events + user info
events_hot_with_rolling = events_hot.withColumn("event_date_num", F.datediff(F.current_date(), F.to_date("event_timestamp"))) \
    .join(daily_counts_hot.select("user_id", "event_date_num", "rolling_90d_events"), on=["user_id", "event_date_num"], how="left") \
    .join(users_df, on="user_id")

In [15]:
# 7. Union hot and cold parts back together
final_df = joined_cold.unionByName(events_hot_with_rolling, allowMissingColumns=True)
final_df.write.csv("events_with_rolling_separate_treatment.csv", header=True, mode="overwrite")

In [16]:
#events_users.groupBy("user_id").count().orderBy(F.col("count").desc()).show(20)

In [17]:
#events_users.write.format("noop").mode("append").save()